# Chapter 6 Computational Lab
## Distribution Functions, Laws and Quantiles

This notebook accompanies Chapter 6 of *Probability Theory with Python and AI*.

The chapter is purely distributional. A measurable observation induces a probability law on the real line; a cdf encodes that law; jumps reveal atoms; and quantiles reverse threshold probabilities through a generalized inverse.

### Learning goals

You should be able to:

1. construct the law $P_X$ of a finite random variable;
2. distinguish equality in law from almost-sure equality;
3. construct and diagnose cdfs;
4. use cdf left limits to compute interval probabilities;
5. explain why a cdf determines the full law;
6. compute survival probabilities and atomic masses;
7. understand why the set of atoms is at most countable;
8. compute quantiles using the generalized inverse;
9. use the correct inequalities $F(q-)\le p\le F(q)$;
10. analyze percentile levels that lie inside jumps;
11. audit AI-generated claims about cdfs and quantiles.

> **Chapter boundary.** Expectation begins in Chapter 7 and is not used here.


## 0. Setup

Finite laws are represented by dictionaries mapping values to exact rational masses.


In [ ]:
from fractions import Fraction
from itertools import combinations
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def fmt_fraction(x):
    x = Fraction(x)
    if x.denominator == 1:
        return str(x.numerator)
    return rf"\frac{{{x.numerator}}}{{{x.denominator}}}"


def finite_law(variable, outcome_masses):
    law = {}
    for outcome, value in variable.items():
        law[value] = law.get(value, Fraction(0, 1)) + outcome_masses[outcome]
    return dict(sorted(law.items()))


def cdf(law, x):
    return sum(
        (p for value, p in law.items() if value <= x),
        Fraction(0, 1),
    )


def cdf_left(law, x):
    return sum(
        (p for value, p in law.items() if value < x),
        Fraction(0, 1),
    )


def survival(law, x):
    return sum(
        (p for value, p in law.items() if value > x),
        Fraction(0, 1),
    )


def quantile(law, p):
    p = Fraction(p)
    if not Fraction(0, 1) < p < Fraction(1, 1):
        raise ValueError("p must lie strictly between 0 and 1.")
    running = Fraction(0, 1)
    for value, mass in sorted(law.items()):
        running += mass
        if running >= p:
            return value
    raise RuntimeError("A normalized law must reach total probability one.")


def show_result(title, *lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))
    for line in lines:
        display(Math(line))
    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Distribution-function tools are ready."
    "</div>"
))


## 1. The law of a random variable

For a real-valued random variable $X$, the law is

$$
P_X(B)=P(X\in B)=P\bigl(X^{-1}(B)\bigr),
\qquad
B\in\mathcal B(\mathbb R).
$$

It is the pushforward of the original probability measure through $X$ and is itself a probability measure on $\mathbb R$.


### Pushing probability to the real line

Let

$$
P(\{\omega_1\})=0.2,\qquad
P(\{\omega_2\})=0.5,\qquad
P(\{\omega_3\})=0.3,
$$

and

$$
X(\omega_1)=0,\qquad
X(\omega_2)=X(\omega_3)=2.
$$

Then

$$
P_X(\{0\})=0.2,\qquad
P_X(\{2\})=0.8.
$$


In [ ]:
outcome_masses = {
    "w1": Fraction(1, 5),
    "w2": Fraction(1, 2),
    "w3": Fraction(3, 10),
}
X = {"w1": 0, "w2": 2, "w3": 2}
law_X = finite_law(X, outcome_masses)

display(Markdown(f"Pushforward law: **{law_X}**"))
display(Math(r"P_X(\{0\})=" + fmt_fraction(law_X[0])))
display(Math(r"P_X(\{2\})=" + fmt_fraction(law_X[2])))
assert sum(law_X.values(), Fraction(0, 1)) == 1


## 2. Same law does not mean same random variable

On

$$
\Omega=\{a,b\},
\qquad
P(\{a\})=P(\{b\})=\frac12,
$$

define

$$
X(a)=0,\quad X(b)=1,
$$

and

$$
Y(a)=1,\quad Y(b)=0.
$$

Both variables have the same law, but they disagree at every outcome. Equality in law does not determine the coupling.


In [ ]:
mass2 = {"a": Fraction(1, 2), "b": Fraction(1, 2)}
X2 = {"a": 0, "b": 1}
Y2 = {"a": 1, "b": 0}

law_X2 = finite_law(X2, mass2)
law_Y2 = finite_law(Y2, mass2)

display(Markdown(f"Same law: **{law_X2 == law_Y2}**"))
display(Markdown(
    f"Agree at any outcome: **{any(X2[w] == Y2[w] for w in X2)}**"
))


## 3. Cumulative distribution functions

The cdf of $X$ is

$$
\boxed{
F_X(x)=P(X\le x)=P_X((-\infty,x]).
}
$$

The law assigns probabilities to all Borel sets; the cdf encodes the same law using closed left half-lines.


In [ ]:
finite_values = widgets.Text(
    value="-1,2",
    description="values",
    layout=widgets.Layout(width="450px"),
)
finite_masses = widgets.Text(
    value="1/4,3/4",
    description="masses",
    layout=widgets.Layout(width="450px"),
)
finite_output = widgets.Output()


def parse_fraction_list(text):
    return [Fraction(x.strip()) for x in text.split(",") if x.strip()]


def update_cdf(*_):
    with finite_output:
        clear_output(wait=True)
        try:
            values = [float(x.strip()) for x in finite_values.value.split(",") if x.strip()]
            masses = parse_fraction_list(finite_masses.value)
        except Exception:
            display(Markdown("**Enter numerical values and rational masses.**"))
            return

        if (
            not values
            or len(values) != len(masses)
            or values != sorted(values)
            or len(values) != len(set(values))
        ):
            display(Markdown("**Use distinct increasing values and the same number of masses.**"))
            return

        if any(p < 0 for p in masses) or sum(masses, Fraction(0, 1)) != 1:
            display(Markdown("**Masses must be non-negative and sum to 1.**"))
            return

        law = dict(zip(values, masses))
        grid = np.linspace(min(values) - 2, max(values) + 2, 700)
        F = [float(cdf(law, x)) for x in grid]

        fig, ax = plt.subplots(figsize=(8, 3.4))
        ax.step(grid, F, where="post")
        ax.set_ylim(-0.03, 1.03)
        ax.set_xlabel("x")
        ax.set_ylabel("F(x)")
        ax.set_title("CDF of a finite law")
        plt.show()


for control in (finite_values, finite_masses):
    control.observe(update_cdf, names="value")

display(widgets.VBox([finite_values, finite_masses, finite_output]))
update_cdf()


## 4. Fundamental properties of a cdf

Every cdf is:

$$
\text{non-decreasing},
$$

$$
\text{right-continuous},
$$

and satisfies

$$
\lim_{x\to-\infty}F_X(x)=0,
\qquad
\lim_{x\to\infty}F_X(x)=1.
$$

Right-continuity follows from continuity of probability from above.


### Two-point example

If

$$
P(X=-1)=\frac14,
\qquad
P(X=2)=\frac34,
$$

then

$$
F_X(x)=
\begin{cases}
0,&x<-1,\\
\frac14,&-1\le x<2,\\
1,&x\ge2.
\end{cases}
$$

The value at a jump is the value **after** the jump.


In [ ]:
two_point = {-1: Fraction(1, 4), 2: Fraction(3, 4)}
grid = [-5, -1, 0, 1.9, 2, 5]
F = [cdf(two_point, x) for x in grid]

display(Markdown(
    "| x | F(x) |\n|---:|---:|\n"
    + "\n".join(f"| {x} | {float(v):.2f} |" for x, v in zip(grid, F))
))
assert all(a <= b for a, b in zip(F, F[1:]))


### Cdf diagnostic dialogue

A proposed function must pass all four checks:

$$
\text{monotonicity},\qquad
\text{right-continuity},\qquad
F(-\infty)=0,\qquad
F(+\infty)=1.
$$


In [ ]:
candidate = widgets.Dropdown(
    options=[
        ("Valid step cdf", "valid"),
        ("Decreases", "decreasing"),
        ("Fails right-continuity", "right"),
        ("Upper limit is 0.8", "upper"),
    ],
    value="valid",
    description="candidate",
)
candidate_output = widgets.Output()


def candidate_value(kind, x):
    if kind == "valid":
        if x < 0:
            return 0.0
        if x < 1:
            return 0.3
        if x < 3:
            return 0.8
        return 1.0

    if kind == "decreasing":
        if x < 0:
            return 0.0
        if x < 1:
            return 0.7
        if x < 3:
            return 0.5
        return 1.0

    if kind == "right":
        if x <= 0:
            return 0.0
        if x <= 2:
            return 0.6
        return 1.0

    if x < 0:
        return 0.0
    return 0.8 * (1 - np.exp(-x))


def update_candidate(*_):
    with candidate_output:
        clear_output(wait=True)
        kind = candidate.value
        grid = np.linspace(-3, 7, 1500)
        values = np.array([candidate_value(kind, x) for x in grid])

        monotone = bool(np.all(np.diff(values) >= -1e-10))
        right_continuous = kind != "right"
        lower_ok = abs(candidate_value(kind, -1000)) < 1e-6
        upper_ok = abs(candidate_value(kind, 1000) - 1) < 1e-6

        display(Markdown(f"**Non-decreasing:** {monotone}"))
        display(Markdown(f"**Right-continuous:** {right_continuous}"))
        display(Markdown(f"**Limit at -infinity is 0:** {lower_ok}"))
        display(Markdown(f"**Limit at +infinity is 1:** {upper_ok}"))

        fig, ax = plt.subplots(figsize=(8, 3.3))
        ax.plot(grid, values)
        ax.set_ylim(-0.05, 1.05)
        ax.set_xlabel("x")
        ax.set_ylabel("F(x)")
        plt.show()


candidate.observe(update_candidate, names="value")
display(widgets.VBox([candidate, candidate_output]))
update_candidate()


## 5. Left limits and interval probabilities

Define

$$
F_X(x-)=\lim_{t\uparrow x}F_X(t)=P(X<x).
$$

For $a<b$,

$$
P(a<X\le b)=F_X(b)-F_X(a),
$$

$$
P(a\le X\le b)=F_X(b)-F_X(a-),
$$

$$
P(a<X<b)=F_X(b-)-F_X(a),
$$

$$
P(a\le X<b)=F_X(b-)-F_X(a-).
$$


In [ ]:
interval_a = widgets.FloatSlider(value=-1, min=-2, max=2, step=1, description="a")
interval_b = widgets.FloatSlider(value=2, min=-1, max=4, step=1, description="b")
interval_output = widgets.Output()


def update_intervals(*_):
    with interval_output:
        clear_output(wait=True)
        a = interval_a.value
        b = interval_b.value
        if a >= b:
            display(Markdown("**Require a<b.**"))
            return

        law = two_point
        display(Math(
            r"P(a<X\le b)=" + fmt_fraction(cdf(law, b) - cdf(law, a))
        ))
        display(Math(
            r"P(a\le X\le b)=" + fmt_fraction(cdf(law, b) - cdf_left(law, a))
        ))
        display(Math(
            r"P(a<X<b)=" + fmt_fraction(cdf_left(law, b) - cdf(law, a))
        ))
        display(Math(
            r"P(a\le X<b)=" + fmt_fraction(cdf_left(law, b) - cdf_left(law, a))
        ))


for control in (interval_a, interval_b):
    control.observe(update_intervals, names="value")

display(widgets.VBox([
    widgets.HBox([interval_a, interval_b]),
    interval_output,
]))
update_intervals()


## 6. A cdf determines the entire law

If

$$
F_X=F_Y,
$$

then

$$
P_X(B)=P_Y(B)
$$

for every Borel set $B$.

The proof uses the $\pi$--$\lambda$ extension mechanism from Chapter 2:

$$
\boxed{
\{(-\infty,x]\}
\longrightarrow
\text{Dynkin class where the laws agree}
\longrightarrow
\mathcal B(\mathbb R).
}
$$

Thus

$$
F_X=F_Y
\iff
P_X=P_Y.
$$


### Finite analogue

For a finite law, every mass is recovered from its cdf jump:

$$
P(X=x)=F_X(x)-F_X(x-).
$$

Recovering all masses recovers the entire finite law.


In [ ]:
law = {
    0: Fraction(1, 4),
    1: Fraction(1, 2),
    4: Fraction(1, 4),
}
recovered = {
    x: cdf(law, x) - cdf_left(law, x)
    for x in law
}

display(Markdown(f"Original law: **{law}**"))
display(Markdown(f"Recovered from cdf: **{recovered}**"))
display(Markdown(f"Exact recovery: **{law == recovered}**"))


## 7. Continuous growth together with atoms

Consider the cdf

$$
F(x)=
\begin{cases}
0,&x<0,\\
0.4x,&0\le x<1,\\
0.7+0.1(x-1),&1\le x<3,\\
1,&x\ge3.
\end{cases}
$$

It has continuous growth on two intervals and jumps at $1$ and $3$.


In [ ]:
def mixed_cdf(x):
    if x < 0:
        return 0.0
    if x < 1:
        return 0.4 * x
    if x < 3:
        return 0.7 + 0.1 * (x - 1)
    return 1.0


xgrid = np.linspace(-1, 4, 1200)
ygrid = [mixed_cdf(x) for x in xgrid]

fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.plot(xgrid, ygrid)
ax.plot(1, 0.4, "o", markerfacecolor="white")
ax.plot(1, 0.7, "o")
ax.plot(3, 0.9, "o", markerfacecolor="white")
ax.plot(3, 1.0, "o")
ax.vlines(1, 0.4, 0.7, linestyles="--")
ax.vlines(3, 0.9, 1.0, linestyles="--")
ax.set_ylim(-0.03, 1.05)
ax.set_xlabel("x")
ax.set_ylabel("F(x)")
ax.set_title("Continuous growth with two jumps")
plt.show()

display(Math(r"P(X=1)=0.7-0.4=0.3"))
display(Math(r"P(X=3)=1-0.9=0.1"))
display(Math(r"P(1<X<3)=0.9-0.7=0.2"))


## 8. Survival function

The survival function is

$$
\boxed{
\overline F_X(x)=P(X>x)=1-F_X(x).
}
$$

The strict event $X>x$ matches the right-continuous convention for $F_X(x)=P(X\le x)$.


In [ ]:
tail_x = widgets.FloatSlider(value=1.0, min=-0.5, max=3.5, step=0.25, description="x")
tail_output = widgets.Output()

def update_tail(*_):
    with tail_output:
        clear_output(wait=True)
        x = tail_x.value
        value = 1 - mixed_cdf(x)
        display(Math(
            r"\overline F(" + f"{x:g}" + r")=1-F(" + f"{x:g}" + r")=" + f"{value:.4f}"
        ))

tail_x.observe(update_tail, names="value")
display(widgets.VBox([tail_x, tail_output]))
update_tail()


## 9. Atoms are exactly the jumps

A point $x$ is an atom when

$$
P(X=x)>0.
$$

Its cdf jump is

$$
\Delta F_X(x)=F_X(x)-F_X(x-).
$$

The exact relation is

$$
\boxed{
P(X=x)=F_X(x)-F_X(x-).
}
$$

Hence the cdf is continuous at $x$ if and only if $P(X=x)=0$.


In [ ]:
jump_data = [
    (0, 0.0, mixed_cdf(0)),
    (1, 0.4, mixed_cdf(1)),
    (2, mixed_cdf(2), mixed_cdf(2)),
    (3, 0.9, mixed_cdf(3)),
]

for x, left, right in jump_data:
    display(Math(
        r"\Delta F(" + str(x) + r")="
        + f"{right:.2f}-{left:.2f}={right-left:.2f}"
    ))


### At most countably many atoms

For

$$
A_n=\left\{x:P(X=x)\ge\frac1n\right\},
$$

each $A_n$ is finite. Every positive mass belongs to some $A_n$, so

$$
A_X=\bigcup_{n=1}^{\infty}A_n
$$

is at most countable.


### Infinitely many atoms are still possible

The law

$$
P(X=k)=2^{-k},
\qquad
k=1,2,\ldots
$$

has one atom at every positive integer.


In [ ]:
atom_N = widgets.IntSlider(value=10, min=1, max=30, description="N")
atom_output = widgets.Output()

def update_atoms(*_):
    with atom_output:
        clear_output(wait=True)
        N = atom_N.value
        ks = np.arange(1, N + 1)
        masses = 2.0 ** (-ks)

        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.bar(ks, masses)
        ax.set_xlabel("k")
        ax.set_ylabel("P(X=k)")
        ax.set_title("First atoms of P(X=k)=2^{-k}")
        plt.show()

        partial = sum(Fraction(1, 2**k) for k in range(1, N + 1))
        remainder = Fraction(1, 2**N)
        display(Math(
            r"\sum_{k=1}^{" + str(N) + r"}2^{-k}=" + fmt_fraction(partial)
        ))
        display(Math(
            r"\text{remaining mass}=" + fmt_fraction(remainder)
        ))

atom_N.observe(update_atoms, names="value")
display(widgets.VBox([atom_N, atom_output]))
update_atoms()


## 10. Quantiles and the generalized inverse

For $0<p<1$,

$$
\boxed{
q_X(p)
=
\inf\{x:F_X(x)\ge p\}.
}
$$

A cdf need not be one-to-one, so this is a generalized inverse.

The universal statement is

$$
\boxed{
F_X(q_X(p)-)\le p\le F_X(q_X(p)).
}
$$

Equivalently,

$$
P(X<q_X(p))
\le p\le
P(X\le q_X(p)).
$$


### Quantile inside a jump

Suppose

$$
P(X=0)=0.6,\qquad
P(X=2)=0.3,\qquad
P(X=5)=0.1.
$$

Then

$$
q_X(p)=
\begin{cases}
0,&0<p\le0.6,\\
2,&0.6<p\le0.9,\\
5,&0.9<p<1.
\end{cases}
$$

At $p=0.75$,

$$
q_X(0.75)=2,
$$

but

$$
F_X(2)=0.9\ne0.75.
$$


In [ ]:
quantile_law = {
    0: Fraction(3, 5),
    2: Fraction(3, 10),
    5: Fraction(1, 10),
}

q_level = widgets.FloatSlider(value=0.75, min=0.01, max=0.99, step=0.01, description="p")
q_output = widgets.Output()

def update_quantile(*_):
    with q_output:
        clear_output(wait=True)
        p = Fraction(str(round(q_level.value, 2)))
        q = quantile(quantile_law, p)
        left = cdf_left(quantile_law, q)
        right = cdf(quantile_law, q)

        display(Math(r"q_X(p)=" + str(q)))
        display(Math(
            fmt_fraction(left) + r"\le" + fmt_fraction(p) + r"\le" + fmt_fraction(right)
        ))

        grid = np.linspace(-1, 6, 900)
        F = [float(cdf(quantile_law, x)) for x in grid]

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.step(grid, F, where="post")
        ax.axhline(float(p), linestyle="--")
        ax.axvline(q, linestyle=":")
        ax.set_ylim(-0.03, 1.03)
        ax.set_xlabel("x")
        ax.set_ylabel("F(x)")
        ax.set_title("Quantile = first threshold reaching p")
        plt.show()

q_level.observe(update_quantile, names="value")
display(widgets.VBox([q_level, q_output]))
update_quantile()


### Quantile monotonicity

If

$$
0<p_1<p_2<1,
$$

then

$$
q_X(p_1)\le q_X(p_2).
$$

The inequality need not be strict. A whole interval of probability levels can share one quantile when the cdf jumps.


In [ ]:
levels = [
    Fraction(1, 10),
    Fraction(1, 2),
    Fraction(3, 5),
    Fraction(3, 4),
    Fraction(9, 10),
    Fraction(19, 20),
]
qs = [quantile(quantile_law, p) for p in levels]

display(Markdown(
    "| p | q_X(p) |\n|---:|---:|\n"
    + "\n".join(
        f"| {float(p):.2f} | {q} |"
        for p, q in zip(levels, qs)
    )
))
assert all(a <= b for a, b in zip(qs, qs[1:]))


### Ordinary inverse case

If the cdf is continuous at $q_X(p)$, then

$$
F_X(q_X(p))=p.
$$

If it is continuous and strictly increasing on all of $\mathbb R$, then the generalized inverse is the ordinary inverse.


In [ ]:
def smooth_cdf(x):
    return 1 / (1 + np.exp(-x))

def smooth_inverse(p):
    return np.log(p / (1 - p))

inv_p = widgets.FloatSlider(value=0.75, min=0.05, max=0.95, step=0.05, description="p")
inv_output = widgets.Output()

def update_inverse(*_):
    with inv_output:
        clear_output(wait=True)
        p = inv_p.value
        q = smooth_inverse(p)
        display(Math(r"q(p)\approx" + f"{q:.6f}"))
        display(Math(r"F(q(p))\approx" + f"{smooth_cdf(q):.6f}"))

inv_p.observe(update_inverse, names="value")
display(widgets.VBox([inv_p, inv_output]))
update_inverse()


## 11. Historical problem: Galton's percentile grades

Consider

$$
F(x)=
\begin{cases}
0,&x<150,\\
0.20,&150\le x<160,\\
0.55,&160\le x<170,\\
0.90,&170\le x<180,\\
1,&x\ge180.
\end{cases}
$$

The generalized inverse gives

$$
q(0.25)=160,\qquad
q(0.50)=160,
$$

$$
q(0.90)=170,\qquad
q(0.95)=180.
$$

Different percentile levels can correspond to the same threshold because a jump crosses many probability levels at once.


In [ ]:
galton_law = {
    150: Fraction(20, 100),
    160: Fraction(35, 100),
    170: Fraction(35, 100),
    180: Fraction(10, 100),
}

for p in [
    Fraction(25, 100),
    Fraction(50, 100),
    Fraction(90, 100),
    Fraction(95, 100),
]:
    display(Math(
        r"q(" + f"{float(p):.2f}" + r")=" + str(quantile(galton_law, p))
    ))


## 12. Recovering a law from a step cdf

Suppose

$$
F(x)=
\begin{cases}
0,&x<0,\\
1/4,&0\le x<1,\\
3/4,&1\le x<4,\\
1,&x\ge4.
\end{cases}
$$

The jump sizes give

$$
P(X=0)=\frac14,\qquad
P(X=1)=\frac12,\qquad
P(X=4)=\frac14.
$$


In [ ]:
step_law = {
    0: Fraction(1, 4),
    1: Fraction(1, 2),
    4: Fraction(1, 4),
}
assert sum(step_law.values(), Fraction(0, 1)) == 1

for x, p in step_law.items():
    display(Math(r"P(X=" + str(x) + r")=" + fmt_fraction(p)))


## 13. A point mass plus probability spread over an interval

Suppose $P(A)=1/3$, $X=0$ on $A$, and on $A^c$,

$$
P(X\le x\mid A^c)=x,
\qquad
0\le x\le1.
$$

Then

$$
F_X(x)=
\begin{cases}
0,&x<0,\\
\frac13+\frac23x,&0\le x<1,\\
1,&x\ge1.
\end{cases}
$$

There is one atom at $0$ of mass $1/3$, and

$$
P(1/4<X\le3/4)=\frac13.
$$


In [ ]:
def point_interval_cdf(x):
    if x < 0:
        return 0.0
    if x < 1:
        return 1/3 + (2/3) * x
    return 1.0

p = point_interval_cdf(3/4) - point_interval_cdf(1/4)
display(Math(r"P(1/4<X\le3/4)=" + f"{p:.6f}"))
assert abs(p - 1/3) < 1e-12


## 14. Exact finite computational verification


In [ ]:
law = {
    0: Fraction(3, 5),
    2: Fraction(3, 10),
    5: Fraction(1, 10),
}

# Normalization.
assert all(p >= 0 for p in law.values())
assert sum(law.values(), Fraction(0, 1)) == 1

# Cdf monotonicity and boundary behavior.
grid = [-2, -1, 0, 1, 2, 3, 4, 5, 6]
F = [cdf(law, x) for x in grid]
assert all(a <= b for a, b in zip(F, F[1:]))
assert cdf(law, -100) == 0
assert cdf(law, 100) == 1

# Jumps recover masses.
for x, mass in law.items():
    assert cdf(law, x) - cdf_left(law, x) == mass

# Survival identity.
for x in grid:
    assert survival(law, x) == 1 - cdf(law, x)

# Four interval formulas.
for a in grid:
    for b in grid:
        if a >= b:
            continue

        p1 = sum((p for x, p in law.items() if a < x <= b), Fraction(0, 1))
        p2 = sum((p for x, p in law.items() if a <= x <= b), Fraction(0, 1))
        p3 = sum((p for x, p in law.items() if a < x < b), Fraction(0, 1))
        p4 = sum((p for x, p in law.items() if a <= x < b), Fraction(0, 1))

        assert p1 == cdf(law, b) - cdf(law, a)
        assert p2 == cdf(law, b) - cdf_left(law, a)
        assert p3 == cdf_left(law, b) - cdf(law, a)
        assert p4 == cdf_left(law, b) - cdf_left(law, a)

# Quantile inequalities and monotonicity.
levels = [Fraction(k, 100) for k in range(1, 100)]
qs = []

for level in levels:
    q = quantile(law, level)
    qs.append(q)
    assert cdf_left(law, q) <= level <= cdf(law, q)

assert all(a <= b for a, b in zip(qs, qs[1:]))

# Jump example.
q = quantile(law, Fraction(3, 4))
assert q == 2
assert cdf_left(law, q) < Fraction(3, 4) < cdf(law, q)

show_result(
    "All Chapter 6 checks passed",
    r"F_X(x)=P(X\le x)",
    r"F_X(x-)=P(X<x)",
    r"P(X=x)=F_X(x)-F_X(x-)",
    r"\overline F_X(x)=1-F_X(x)",
    r"F_X(q_X(p)-)\le p\le F_X(q_X(p))",
    note="No expectation is used."
)


## 15. Guided exercise generator


In [ ]:
rng = random.Random(20260815)

kind = widgets.Dropdown(
    options=[
        ("Random", "random"),
        ("Law", "law"),
        ("Jump", "jump"),
        ("Interval", "interval"),
        ("Survival", "survival"),
        ("Quantile", "quantile"),
        ("Same law", "same"),
    ],
    value="random",
    description="Type",
)
new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer = widgets.Text(description="Answer")
prompt_out = widgets.Output()
feedback_out = widgets.Output()
state = {}


def make_exercise(_=None):
    k = kind.value
    if k == "random":
        k = rng.choice(["law", "jump", "interval", "survival", "quantile", "same"])

    if k == "law":
        target = "0.8"
        prompt = "Outcome masses are 0.2, 0.5, 0.3 and X-values are 0,2,2. Find P_X({2})."
        hint = "Add all outcome masses mapped to 2."
        solution = r"P_X(\{2\})=0.8."

    elif k == "jump":
        target = "0.3"
        prompt = "If F(1-)=0.4 and F(1)=0.7, find P(X=1)."
        hint = "Point mass equals jump size."
        solution = r"P(X=1)=0.3."

    elif k == "interval":
        target = "0.4"
        prompt = "If F(a)=0.2 and F(b)=0.6, find P(a<X<=b)."
        hint = "Use F(b)-F(a)."
        solution = r"P(a<X\le b)=0.4."

    elif k == "survival":
        target = "0.3"
        prompt = "If F(1)=0.7, find P(X>1)."
        hint = "Use 1-F(1)."
        solution = r"P(X>1)=0.3."

    elif k == "quantile":
        target = "2"
        prompt = "Masses at 0,2,5 are 0.6,0.3,0.1. Find q_X(0.75)."
        hint = "Find the first point where the cdf reaches 0.75."
        solution = r"q_X(0.75)=2."

    else:
        target = "no"
        prompt = "If X and Y have the same law, must they be almost surely equal? yes/no"
        hint = "The law does not determine the coupling."
        solution = r"\text{No.}"

    state.clear()
    state.update(target=target, hint=hint, solution=solution)
    answer.value = ""

    with prompt_out:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))
    with feedback_out:
        clear_output(wait=True)


def show_hint(_):
    with feedback_out:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))


def reveal(_):
    with feedback_out:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_out:
        clear_output(wait=True)
        guess = answer.value.strip().lower().replace(" ", "")
        target = state["target"].replace(" ", "")
        try:
            if target != "no" and abs(float(guess) - float(target)) < 1e-9:
                display(Markdown("**Correct.**"))
                return
        except Exception:
            pass
        display(Markdown("**Correct.**" if guess == target else "**Not yet.**"))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([kind, new_button]),
    prompt_out,
    widgets.HBox([answer, check_button]),
    widgets.HBox([hint_button, reveal_button]),
    feedback_out,
]))
make_exercise()


## 16. AI Audit: laws, cdfs and quantiles

Audit every AI-generated argument using the following questions:

1. Does a proposed finite law have non-negative masses summing to one?
2. Is $P_X(B)$ correctly defined through the inverse image?
3. Is equality in law being confused with almost-sure equality?
4. Is a proposed cdf non-decreasing?
5. Is it right-continuous?
6. Are the two boundary limits correct?
7. Are left limits used correctly at open/closed interval endpoints?
8. Is an atom correctly identified as a jump?
9. Is the survival event $X>x$, not $X\ge x$?
10. Is the AI falsely claiming that every cdf is differentiable?
11. Is it falsely allowing uncountably many atoms?
12. Is the generalized inverse being confused with an ordinary inverse?
13. Is $F(q_X(p))=p$ being asserted at a jump?
14. Is $q_X$ claimed to be strictly increasing rather than non-decreasing?
15. Is the cdf-uniqueness theorem correctly interpreted as equality of laws?

### Claims to audit

- “Every cdf is differentiable.”
- “If $0<p_1<p_2<1$, then $q_X(p_1)<q_X(p_2)$.”
- “Whenever $0<F_X(x)<1$, one always has $q_X(F_X(x))=x$.”

All three are false in general.


### Suggested AI audit prompts

- “Generate four candidate cdfs and make me diagnose each structural property.”
- “Give me a step cdf with four jumps and ask me to recover all atomic masses.”
- “Choose a probability level inside a jump and require verification of $F(q-)\le p\le F(q)$.”
- “Explain why agreement on closed left rays extends to equality on every Borel set.”
- “Construct same-law random variables that disagree at every outcome.”


## 17. Self-check quiz


In [ ]:
quiz_data = [
    ("1. The law of X is:", ["Choose...", "a probability measure on B(R)", "the same function as X"], "a probability measure on B(R)", r"P_X(B)=P(X\in B)."),
    ("2. Equal laws imply a.s. equality:", ["Choose...", "true", "false"], "false", r"\text{The coupling is not determined.}"),
    ("3. Every cdf is:", ["Choose...", "non-decreasing and right-continuous", "strictly increasing", "differentiable"], "non-decreasing and right-continuous", r"\text{Jumps and flat portions are allowed.}"),
    ("4. F_X(x-) equals:", ["Choose...", "P(X<x)", "P(X<=x)", "P(X>x)"], "P(X<x)", r"F_X(x-)=P(X<x)."),
    ("5. P(X=x) equals:", ["Choose...", "F(x)-F(x-)", "F(x)", "1-F(x)"], "F(x)-F(x-)", r"P(X=x)=\Delta F_X(x)."),
    ("6. A continuous cdf has:", ["Choose...", "no atoms", "one atom", "uncountably many atoms"], "no atoms", r"\text{Atoms are exactly jumps.}"),
    ("7. A cdf can have uncountably many atoms:", ["Choose...", "true", "false"], "false", r"\text{Atoms are at most countable.}"),
    ("8. The survival function is:", ["Choose...", "P(X>x)=1-F(x)", "P(X>=x)=1-F(x)", "F(x-)"], "P(X>x)=1-F(x)", r"\overline F_X(x)=1-F_X(x)."),
    ("9. Universal quantile statement:", ["Choose...", "F(q)=p", "F(q-)<=p<=F(q)", "q is strictly increasing"], "F(q-)<=p<=F(q)", r"F_X(q_X(p)-)\le p\le F_X(q_X(p))."),
    ("10. A cdf determines:", ["Choose...", "the entire law", "the coupling with every variable"], "the entire law", r"\text{Agreement of cdfs means agreement of laws.}"),
]

quiz_widgets = []
rows = []
for prompt, options, _, _ in quiz_data:
    d = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="390px"),
    )
    quiz_widgets.append(d)
    rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:660px'>{prompt}</div>"),
        d,
    ]))

grade = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()

def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)
        score = sum(
            w.value == correct
            for w, (_, _, correct, _) in zip(quiz_widgets, quiz_data)
        )
        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))
        for i, (w, (_, _, correct, explanation)) in enumerate(zip(quiz_widgets, quiz_data), 1):
            mark = "✓" if w.value == correct else "✗"
            display(Markdown(f"**{mark} Question {i}:** `{correct}`"))
            display(Math(explanation))

grade.on_click(grade_quiz)
display(widgets.VBox(rows + [grade, quiz_output]))


## 18. Chapter map

| Chapter concept | Computational representation |
|---|---|
| law $P_X$ | finite pushforward |
| equality in law | same-law/different-coupling example |
| cdf | exact cumulative mass |
| cdf properties | structural diagnostic widget |
| left limit | $P(X<x)$ |
| interval probabilities | four endpoint conventions |
| cdf determines law | jump recovery plus $\pi$--$\lambda$ interpretation |
| mixed cdf | continuous growth with two jumps |
| survival function | $1-F(x)$ |
| atoms | cdf jump sizes |
| countably many atoms | infinite geometric-atom example |
| quantile | generalized inverse |
| quantile inequalities | interactive jump-crossing example |
| ordinary inverse case | smooth strictly increasing illustration |
| percentiles | Galton jump example |
| AI Audit | structural diagnostics |

> The law is the induced probability model on the real line; the cdf encodes that law; jumps reveal atoms; and quantiles invert the cdf in the generalized sense.
